## Description
This notebook runs the model for scenarios specified inside the model_run function in model.py

### Imports

In [6]:
import pandas as pd
import numpy as np
import pylab as plt

import pypsa

from pathlib import Path
import gc
from itertools import product

# Import the public function from the package you completed
from energy_model import model_run

### Setting model parameters

In [10]:
# Default parameters

# model years
years = [2020, 2030, 2040, 2050]

# intra-year temporal resolution
t_res = 7 # hours

# discount rates
rt = 0.07 # technical discount rate
rs = 0.01 # social discount rate

# passenger cars conversion factor
passenger_cars_cf = 40*0.1  # km/h

# Define where the files are searched from
db_dir = "input_databases/default"

### Setting scenario parameters

In [11]:
# Scenario definitions: 2050 endpoint multipliers (1.0 in 2020, linear to 2050).
# Given our preprocessing functions, we only have to multiply the 2050 values
# since 2030 and 2040 will be interpolated linearly. The specific multipliers
# are based on data sources specified in the final report.
scenario_definitions = {
    "default_tech_low_carbon": {
        "capex_2050": 1.0,
        "efficiency_2050": 1.0,
        "carbon_2050": 1.0,
    },
    "default_tech_high_carbon": {
        "capex_2050": 1.0,
        "efficiency_2050": 1.0,
        "carbon_2050": 12.0,
    },
    "improved_tech_low_carbon": {
        "capex_2050": 0.8,
        "efficiency_2050": 1.12,
        "carbon_2050": 1.0,
    },
    "improved_tech_high_carbon": {
        "capex_2050": 0.8,
        "efficiency_2050": 1.12,
        "carbon_2050": 12.0,
    },
}

### Running functions

In [12]:
def run_scenario(
    out_dir: Path = Path("results/temporal_scenarios"),
    scenario: str = "default_tech_low_carbon",
    scenario_settings: dict = None,
    overwrite: bool = False,
) -> None:
    """Solve one scenario and export it; optionally skip existing results."""
    if scenario_settings is None:
        scenario_settings = scenario_definitions[scenario]

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # model.py writes its solver log to results/solver_run.log.
    Path("results").mkdir(parents=True, exist_ok=True)

    out_file = out_dir / f"{scenario}.nc"

    if out_file.exists() and not overwrite:
        print(f"{scenario} already done, skipping")
        return

    print(f"Running {scenario} with settings: {scenario_settings}")

    n = model_run(
        db_dir=db_dir,
        years=years,
        t_res=t_res,
        passenger_cars_cf=passenger_cars_cf,
        rt=rt,
        rs=rs,
        scenario_settings=scenario_settings,
    )

    n.name = scenario
    n.export_to_netcdf(out_file)

    print(f"Saved {out_file}")


def save_run(
    out_dir: Path = Path("results/temporal_scenarios"),
    scenario: str = "default_tech_low_carbon",
    scenario_settings: dict = None,
    overwrite: bool = False,
) -> None:
    """Run one scenario; print and continue if it fails."""
    try:
        run_scenario(
            out_dir=out_dir,
            scenario=scenario,
            scenario_settings=scenario_settings,
            overwrite=overwrite,
        )
    except Exception as e:
        print(f"{scenario} failed: {e}")
    finally:
        gc.collect()

### Run the model

In [13]:
# Four scenarios with year-dependent multipliers.

final_results = Path("results/temporal_scenarios")

for scenario, settings in scenario_definitions.items():
    save_run(
        out_dir=final_results,
        scenario=scenario,
        scenario_settings=settings,
        overwrite=True,
    )

Running default_tech_low_carbon with settings: {'capex_2050': 1.0, 'efficiency_2050': 1.0, 'carbon_2050': 1.0}
2020
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 316.26it/s]


Set parameter Username
Set parameter LicenseID to value 2858749
Academic license - for non-commercial use only - expires 2027-08-31
Read LP format model from file /private/var/folders/fx/w8lwd2d95938qx99fyfp5kw00000gp/T/linopy-problem-19qmdwdf.lp
Reading time = 1.40 seconds
obj: 720020 rows, 334391 columns, 1694694 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2195108239281.4597
Saved results/temporal_scenarios/default_tech_low_carbon.nc
Running default_tech_high_carbon with settings: {'capex_2050': 1.0, 'efficiency_2050': 1.0, 'carbon_2050': 12.0}
2020
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 284.25it/s]

Set parameter Username
Set parameter LicenseID to value 2858749
Academic license - for non-commercial use only - expires 2027-08-31


...

Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 285.91it/s]


Set parameter Username
Set parameter LicenseID to value 2858749
Academic license - for non-commercial use only - expires 2027-08-31
Read LP format model from file /private/var/folders/fx/w8lwd2d95938qx99fyfp5kw00000gp/T/linopy-problem-xrdwe1wi.lp
Reading time = 1.40 seconds
obj: 720020 rows, 334391 columns, 1694694 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2194219987862.5244
Saved results/temporal_scenarios/improved_tech_low_carbon.nc
Running improved_tech_high_carbon with settings: {'capex_2050': 0.8, 'efficiency_2050': 1.12, 'carbon_2050': 12.0}
2020
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 291.36it/s]

Set parameter Username
Set parameter LicenseID to value 2858749
Academic license - for non-commercial use only - expires 2027-08-31


Read LP format model from file /private/var/folders/fx/w8lwd2d95938qx99fyfp5kw00000gp/T/linopy-problem-z8570eax.lp
Reading time = 1.45 seconds
obj: 720020 rows, 334391 columns, 1694694 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2343094752941.4346
Saved results/temporal_scenarios/improved_tech_high_carbon.nc
